# 02 · Data Preprocessing & Feature Engineering
Cleans the weekly surveillance data, joins weather, engineers the features the
models consume (lags, rolling statistics, cyclical week-of-year, weather, state
encoding, seasonal deviation), assigns a strictly temporal train/val/test split,
fits and saves a per-disease feature scaler, and writes a processed table per
disease for the model notebooks.

**Split (per the project spec):** train = 2015–2021, validation = 2022,
test = 2023–2024.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

DATA = Path("../backend/data")
MODELS = Path("../backend/models")
MODELS.mkdir(parents=True, exist_ok=True)

DISEASES = ["lassa", "cholera", "meningitis", "mpox"]
LAGS = [1, 2, 3, 4, 8, 12]
FEATURES = ([f"lag_{l}" for l in LAGS] +
            ["roll4_mean", "roll4_std", "seasonal_dev",
             "woy_sin", "woy_cos", "temp_mean", "rainfall_sum", "state_code"])
TRAIN_YEARS = range(2015, 2022)   # 2015–2021
VAL_YEAR = 2022
TEST_YEARS = (2023, 2024)

## Load cases + weather and merge
Weather is joined on (year, week, state); any gaps are filled with the national
weekly mean so every row has weather features.

In [2]:
weather = pd.read_csv(DATA / "weather.csv")
print("weather:", weather.shape, "| states:", weather.state.nunique())


def build(disease: str) -> pd.DataFrame:
    df = pd.read_csv(DATA / f"ncdc_{disease}.csv", parse_dates=["date"])
    df = df.merge(weather, on=["year", "week", "state"], how="left")
    for col in ["temp_mean", "rainfall_sum"]:
        df[col] = df[col].fillna(df.groupby("week")[col].transform("mean"))
    return df.sort_values(["state", "date"]).reset_index(drop=True)

weather: (18756, 5) | states: 36


## Feature engineering (within each state)
All temporal features are computed per state to avoid leakage across states, and
rolling/lag features use only past weeks (shifted) so no future information leaks
into a row's predictors.

In [3]:
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("state", group_keys=False)
    for l in LAGS:
        df[f"lag_{l}"] = g["confirmed"].shift(l)
    df["roll4_mean"] = g["confirmed"].transform(lambda s: s.shift(1).rolling(4).mean())
    df["roll4_std"] = g["confirmed"].transform(lambda s: s.shift(1).rolling(4).std())
    wk_mean = df.groupby(["state", "week"])["confirmed"].transform("mean")
    df["seasonal_dev"] = df["confirmed"] - wk_mean          # deviation from typical week
    df["woy_sin"] = np.sin(2 * np.pi * df["week"] / 52)
    df["woy_cos"] = np.cos(2 * np.pi * df["week"] / 52)
    df["state_code"] = df["state"].astype("category").cat.codes
    return df


def split_of(year: int) -> str:
    if year in TRAIN_YEARS:
        return "train"
    return "val" if year == VAL_YEAR else "test"


processed = {}
for d in DISEASES:
    df = engineer(build(d))
    df["split"] = df["year"].map(split_of)
    df = df.dropna(subset=[f"lag_{max(LAGS)}", "roll4_std"]).reset_index(drop=True)
    processed[d] = df
    print(f"{d:11s} rows={len(df):6d}  "
          f"train={(df.split=='train').sum():5d}  "
          f"val={(df.split=='val').sum():4d}  "
          f"test={(df.split=='test').sum():5d}")

lassa       rows= 18796  train=13024  val=1924  test= 3848


cholera     rows= 18796  train=13024  val=1924  test= 3848


meningitis  rows= 18796  train=13024  val=1924  test= 3848


mpox        rows= 14948  train= 9176  val=1924  test= 3848


## Normalisation
A `StandardScaler` is fit on **training rows only** (no leakage) and saved per
disease; the model notebooks reuse it. The processed table itself is saved
unscaled so it stays interpretable for evaluation and the dashboard.

In [4]:
for d in DISEASES:
    df = processed[d]
    scaler = StandardScaler().fit(df.loc[df.split == "train", FEATURES])
    joblib.dump({"scaler": scaler, "features": FEATURES}, MODELS / f"scaler_{d}.pkl")
    df.to_csv(DATA / f"processed_{d}.csv", index=False)
    print(f"saved processed_{d}.csv  +  scaler_{d}.pkl  ({len(FEATURES)} features)")

saved processed_lassa.csv  +  scaler_lassa.pkl  (14 features)


saved processed_cholera.csv  +  scaler_cholera.pkl  (14 features)


saved processed_meningitis.csv  +  scaler_meningitis.pkl  (14 features)


saved processed_mpox.csv  +  scaler_mpox.pkl  (14 features)


## Feature preview (Lassa)

In [5]:
processed["lassa"][["date", "state", "confirmed"] + FEATURES].head(8)

,date,state,confirmed,lag_1,lag_2,lag_3,lag_4,lag_8,lag_12,roll4_mean,roll4_std,seasonal_dev,woy_sin,woy_cos,temp_mean,rainfall_sum,state_code
0,2015-03-23,Abia,2,0.0,3.0,1.0,1.0,1.0,1.0,1.25,1.258306,0.5,1.000000,-1.608123e-16,25.971429,74.1,0
1,2015-03-30,Abia,0,2.0,0.0,3.0,1.0,3.0,1.0,1.50,1.290994,-0.9,0.992709,-1.205367e-01,26.271429,33.1,0
2,2015-04-06,Abia,0,0.0,2.0,0.0,3.0,2.0,0.0,1.25,1.500000,-0.7,0.970942,-2.393157e-01,26.157143,34.3,0
3,2015-04-13,Abia,0,0.0,0.0,2.0,0.0,0.0,1.0,0.50,1.000000,-0.4,0.935016,-3.546049e-01,27.342857,3.5,0
4,2015-04-20,Abia,0,0.0,0.0,0.0,2.0,1.0,1.0,0.50,1.000000,-0.5,0.885456,-4.647232e-01,27.000000,19.2,0
5,2015-04-27,Abia,1,0.0,0.0,0.0,0.0,1.0,3.0,0.00,0.000000,0.6,0.822984,-5.680647e-01,26.757143,19.8,0
6,2015-05-04,Abia,0,1.0,0.0,0.0,0.0,3.0,2.0,0.25,0.500000,-0.2,0.748511,-6.631227e-01,26.871429,10.6,0
7,2015-05-11,Abia,2,0.0,1.0,0.0,0.0,0.0,0.0,0.25,0.500000,1.4,0.663123,-7.485107e-01,25.828571,79.5,0


### Takeaways
- Weather joined and gap-filled; all temporal features computed within each
  state with past-only windows (no leakage); lag warm-up rows dropped.
- Split is strictly temporal (2015–21 / 2022 / 2023–24), so the test set
  simulates true forecasting on unseen future weeks.
- One scaler per disease, fit on training rows only, saved for the model stage.

**Next:** `03 · LSTM training` — sequence construction and 4-week-ahead forecasting.